# CS-DFA — Zero-Shot Cross-Country Detection of Information Operations

**Colab GPU walkthrough** for our Social Media Mining (NCU CE7066) final project,
which extends **IOHunter** (AAAI 2025) with **CS-DFA**: channel-specific GNNs +
coverage-gated fusion for zero-shot cross-country influence-operation account
detection.

- **Repo & paper:** https://github.com/NelsonKCT/SMM_Final_Project (paper in `paper/`)
- **Data** (~3.6 GB, the six-country IOHunter release) lives on Google Drive and is
  symlinked in; **code** is cloned fresh from GitHub every session.
- **What this notebook covers:** environment setup → data sanity checks → a smoke
  test → the six-country CS-DFA main experiment → the three mechanism ablations →
  the 10%/50% source-data-efficiency runs → a final results summary.
- **Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU). One six-country
  batch takes ≈ 20–40 min.

The training cells write per-country logs to `results/` on Drive, so every number
in the paper can be traced back to a log file produced here.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

The dataset (and the `results/` log directory) live on Drive; after authorizing,
Drive appears under `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Project paths

`PROJECT_DIR` must contain `data/processed/<country>/` for the six countries
(china, iran, UAE, cuba, russia, venezuela).

In [ ]:
import os

# Point this at the Drive folder that holds data/ (code is cloned by the next
# section and does not need to live on Drive)
PROJECT_DIR = '/content/drive/MyDrive/SMM_Final_Project'

assert os.path.isdir(os.path.join(PROJECT_DIR, 'data', 'processed')), \
    f'data/processed/ not found — upload the dataset to: {PROJECT_DIR}'
print('data path OK')
print('country folders:', sorted(os.listdir(os.path.join(PROJECT_DIR, 'data', 'processed'))))

## 4. Get the latest code from GitHub

Code (git) and data (Drive) are kept separate: each session clones the latest
`main` to local disk, then symlinks `data/` and `results/` back to Drive — data
is too large to copy, and logs written to `results/` survive the session.

In [ ]:
%cd /content
import os, shutil

REPO_URL  = 'https://github.com/NelsonKCT/SMM_Final_Project.git'
BRANCH    = 'main'
LOCAL_DIR = '/content/smm'

# For a private repo, use a token URL instead (create a PAT under GitHub
# Settings -> Developer settings):
# REPO_URL = 'https://<USER>:<PAT>@github.com/NelsonKCT/SMM_Final_Project.git'

# 1) Remove any stale copy and clone the latest code locally
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
!git clone -b {BRANCH} {REPO_URL} {LOCAL_DIR}

# Stop right away if the clone failed, before the symlinks hide the problem
assert os.path.isdir(f'{LOCAL_DIR}/src'), 'clone failed — check the git output above'

# 2) data is too large to copy: symlink it back to Drive (read-only)
if os.path.lexists(f'{LOCAL_DIR}/data'):
    (os.remove if os.path.islink(f'{LOCAL_DIR}/data') else shutil.rmtree)(f'{LOCAL_DIR}/data')
os.symlink(f'{PROJECT_DIR}/data', f'{LOCAL_DIR}/data')

# 3) results is symlinked to Drive too, so finished logs survive disconnects
for subdir in ['results', 'results/tset_logs', 'results/csdfa_logs', 'results/diagnostics']:
    os.makedirs(f'{PROJECT_DIR}/{subdir}', exist_ok=True)
if os.path.lexists(f'{LOCAL_DIR}/results'):
    (os.remove if os.path.islink(f'{LOCAL_DIR}/results') else shutil.rmtree)(f'{LOCAL_DIR}/results')
os.symlink(f'{PROJECT_DIR}/results', f'{LOCAL_DIR}/results')

RUN_DIR = LOCAL_DIR

# 4) Verify: the checked-out commit + key files/folders exist
print('\n--- verify ---')
!cd {RUN_DIR} && git log --oneline -1
print('run_experiments.py exists:', os.path.exists(f'{RUN_DIR}/src/run_experiments.py'))
print('CSDFA script exists:', os.path.exists(f'{RUN_DIR}/src/run_MultiModalGNN_CrossAttention_CrossCountry_CSDFA.py'))
print('tset_logs dir：', os.path.isdir(f'{PROJECT_DIR}/results/tset_logs'))
print('csdfa_logs dir：', os.path.isdir(f'{PROJECT_DIR}/results/csdfa_logs'))
print('RUN_DIR:', RUN_DIR, '->', sorted(os.listdir(RUN_DIR)))


## 5. Install dependencies

Colab pre-installs CUDA-enabled torch; we only add `torch_geometric` and
`mlflow`. (No compiled `torch_scatter` needed for GCN/SAGE/GAT in recent
PyG versions.)

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

# Only torch_geometric and mlflow are needed.
# Do NOT install node2vec: it pulls in gensim, downgrades numpy below 2,
# and breaks binary compatibility with Colab's numpy 2.x. Nothing here needs
# it: my_utils.py imports it lazily (only the Node2Vec baseline uses it).
!pip install -q torch_geometric mlflow
# networkx / scikit-learn / scipy / pandas ship with Colab already

import torch_geometric, mlflow, networkx, sklearn
print('torch_geometric', torch_geometric.__version__, '— dependencies OK')

## 6. Smoke test — one country through the full pipeline

Runs CS-DFA with Iran as the zero-shot target (smallest data). Verifies the GPU
pipeline end-to-end in a few minutes; expect a final `[TEST] f1_macro` around
0.99 in the log tail.

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa --countries iran --device 0


## 7. CS-DFA — Channel-Specific DFA with Coverage-Gated Fusion

The five behavioral sub-networks (`coRT`, `coURL`, `hashSeq`, `fastRT`,
`tweetSim`) cover very different fractions of accounts per country (China: three
channels at 3–7%). CS-DFA therefore:

- runs **one GNN per sub-network** instead of merging them into a single graph;
- fuses the per-channel embeddings with **coverage-gated attention** — a binary
  participation mask sets the score of channels an account does not belong to
  to −∞, so isolated-node noise never enters the fused representation;
- adds an optional coverage prior `λ·log(coverage)` and channel-level CORAL on
  the stable channels (both shown to be **inert** by the ablations below);
- keeps BCE loss (an earlier diagnosis showed Focal loss hurts extreme-imbalance
  targets like Cuba).

### 7.1 Data sanity check — precomputed edge indices

Each country needs `edge_index_<subnet>.th` for all five sub-networks (computing
them from scratch on the first run is very slow). Expect `5/5` for all six
countries.

In [ ]:
import glob
for c in ['china', 'iran', 'UAE', 'cuba', 'russia', 'venezuela']:
    files = sorted(glob.glob(f'{PROJECT_DIR}/data/processed/{c}/edge_index_*.th'))
    names = [f.split('edge_index_')[-1].replace('.th', '') for f in files]
    print(f"{c:11s} {len(files)}/5  {names}")


### 7.2 Main experiment — all six countries

Zero-shot protocol: for each target country, train on the other five and
evaluate on the held-out target test split; 5 random splits, mean ± std
`f1_macro`. Logs go to `results/csdfa_logs/` on Drive. Expect CS-DFA to beat the
DFA baseline (avg 0.775) on every country, with the largest gains on the
low-coverage targets (China, Venezuela).

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa --device 0


### 7.3 Ablations — which mechanism actually matters?

Three controls, in decreasing order of importance:

1. `csdfa_nogate` — **the decisive control**: gating off (+ prior off + CORAL
   off) = a bare five-channel backbone. If coverage-gated fusion is the real
   contribution, this should collapse. *(It does: avg 0.952 → 0.686, below the
   merged-graph DFA baseline.)*
2. `csdfa_nocoral` — CORAL off. *(Changes the average by 0.002 → inert.)*
3. `csdfa_noprior` — coverage prior λ=0. *(Changes it by 0.000 → inert.)*

In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa_nocoral --device 0


In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa_noprior --device 0


In [ ]:
%cd {RUN_DIR}/src
!python run_experiments.py --method csdfa_nogate --device 0


### 7.4 Source-data efficiency — 10% / 50% of the labeled source accounts

How much labeled source data does CS-DFA actually need? `--source_frac f` keeps a
fixed, class-stratified fraction `f` of each source country's labeled accounts
(target data and evaluation splits are untouched; `[SourceFrac]` audit lines in
the log show exactly what was kept). We rerun CS-DFA and the DFA baseline at 10%
and 50% and compare with the 100% runs above. Expect CS-DFA to stay close to its
full-data performance even at 10%.

In [ ]:
%cd /content/smm/src
!python run_experiments.py --method csdfa --source_frac 0.1 --device 0
!python run_experiments.py --method csdfa --source_frac 0.5 --device 0
!python run_experiments.py --method dfa --source_frac 0.1 --device 0
!python run_experiments.py --method dfa --source_frac 0.5 --device 0

## 8. Results summary — parse every log

Reads all final logs from `results/` on Drive and prints the tables the paper
uses: the six-country CS-DFA main result (overall + per-subnet slices) and the
10%/50%/100% source-data-efficiency comparison for both CS-DFA and DFA.

In [ ]:
import glob, re

RESULTS = f'{PROJECT_DIR}/results'

def grab_f1(path):
    txt = open(path, encoding='utf-8', errors='replace').read()
    m = re.search(r'\[TEST\] f1_macro:\s*([\d.]+)(?:\+-([\d.]+))?', txt)
    return (float(m.group(1)), float(m.group(2) or 0)) if m else None

COUNTRIES = ['china', 'iran', 'UAE', 'cuba', 'russia', 'venezuela']

def table(title, pattern):
    rows = {}
    for f in glob.glob(pattern, recursive=True):
        c = re.search(r'_(\w+)\.txt$', f).group(1)
        rows[c] = grab_f1(f)
    if not rows:
        return
    print(f'\n{title}')
    vals = []
    for c in COUNTRIES:
        if c in rows and rows[c]:
            m, s = rows[c]
            vals.append(m)
            print(f'  {c:10s} {m:.3f} +- {s:.3f}')
    if vals:
        print(f'  {"AVERAGE":10s} {sum(vals)/len(vals):.3f}   (n={len(vals)})')

table('CS-DFA main (100% source data)', f'{RESULTS}/**/zero-shot_CSDFA_[!f]*.txt')
table('CS-DFA ablation: no CORAL',      f'{RESULTS}/**/zero-shot_CSDFAnocoral_*.txt')
table('CS-DFA ablation: no prior',      f'{RESULTS}/**/zero-shot_CSDFAnoprior_*.txt')
table('CS-DFA ablation: no gate',       f'{RESULTS}/**/zero-shot_CSDFAnogate_*.txt')
table('CS-DFA @ 10% source data',       f'{RESULTS}/csdfa_logs/zero-shot_CSDFA_frac10_*.txt')
table('CS-DFA @ 50% source data',       f'{RESULTS}/csdfa_logs/zero-shot_CSDFA_frac50_*.txt')
table('DFA    @ 10% source data',       f'{RESULTS}/dfa_logs/zero-shot_DFA_frac10_*.txt')
table('DFA    @ 50% source data',       f'{RESULTS}/dfa_logs/zero-shot_DFA_frac50_*.txt')